# Extract CellViT nucleus features

Preflight runs first: one real batch through the exact checkpoint, postprocessor,
DINO crop encoder and cache writer, plus a runtime estimate. Full extraction
starts only if that passes, so a bad path or scale does not burn a session.

**Both T4s are used.** The patch range is split into contiguous blocks of *whole
batches*, one worker process per GPU, each writing its own per-batch `.npz` files
into the shared `.build` directory. A final single-process pass merges them into
the cache layout the sampler reads.

Why whole batches, never a split one: `CellViTPatchExtractor._prepare_batch` pads
every image in a batch up to the largest one in that batch, so a batch with
different members is a different forward pass on any variable-size dataset.
Assigning whole batches means each batch has exactly the members it would have
had in a one-GPU run.

**Seeding.** Each shard process passes the same `--seed`, and `get_data_loaders`
derives the train/test split from it. Two workers with different seeds would
shard two different splits. The cache directory name carries the seed, and the
manifest carries a fingerprint of the sample order, which the sampler verifies
against its own split before using the cache.

**Resume.** A batch file that already exists is not recomputed, so a session
that dies part-way — or hits the 12-hour limit — can simply be re-run.

Needs `requirements-cellvit.txt` (numba / cv2 / scikit-image). No other notebook
does.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

# The nucleus notebook installs ONLY its own two requirement files. It does not
# install requirements.txt: that one carries matplotlib, seaborn, scikit-learn
# and open_clip_torch for the sampler and evaluation notebooks, and nothing on
# the extraction path imports them.
#
# Two commands, not one, because --require-hashes applies to a whole file. The
# CellViT wheel is hash-pinned; PyYAML and Pillow are C extensions with a
# different wheel (and hash) per platform, so hashing them would break on the
# next Kaggle image bump.
#
# --no-deps is load-bearing. The cellvit wheel declares 22 dependencies, 19 of
# them unpinned (colour, colorama, ray, pathopatch,
# opencv-python-headless==4.7.0.72, numpy<2 ...):
#
#   * with a --hash present, pip demands == pins for every transitive dependency
#     too, and aborts on the first one that lacks it:
#         ERROR: In --require-hashes mode, all requirements must have their
#         versions pinned with ==. These do not: colour ...
#   * and even pinned, that tree would replace Kaggle's NumPy, OpenCV, Pydantic
#     and Ray. This adapter runs CellViT256 over patches and imports none of the
#     WSI pipeline those pins exist for.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "--require-hashes", "-r", "requirements-cellvit.txt",
])

# The rest, with their own dependencies. --no-deps above means CellViT brought
# none of its own, so einops in particular has to come from here.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-cellvit-extra.txt",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub", "hf-transfer",
])

# torch, torchvision, numpy, scipy, pandas, scikit-image, cv2 and numba are
# preinstalled on Kaggle and are NOT touched: reinstalling them risks an ABI
# break under torch. preflight_cellvit.py imports each one and names the
# offender, which is what should decide whether the image needs changing.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")
print("dependency setup complete")


In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
# ---- EDIT ONLY THIS CELL ----
DATASET = "pathmnist"          # pathmnist | skintissue; HistoSet needs per-source MPP
SEED = 42
DATA_PATH = DATA_PATHS[DATASET]

CHECKPOINT_CANDIDATES = [
    DATA_ROOT / "CellViT-256-x40-AMP.pth",
    Path("/kaggle/input/cellvit-checkpoints/CellViT-256-x40-AMP.pth"),
]
CHECKPOINT_PATH = str(
    next((p for p in CHECKPOINT_CANDIDATES if p.is_file()), CHECKPOINT_CANDIDATES[0])
)

# PathMNIST source pixels are 0.5 MPP. MODEL_MPP/MAGNIFICATION must match the checkpoint.
INPUT_MPP = 0.5
MODEL_MPP = 0.25
MAGNIFICATION = 40

CACHE_DIR = "/kaggle/working/cellvit_features"
DINO_MODEL = "facebook/dinov2-base"   # or a mounted local model directory
BATCH_SIZE = 2
DINO_CROP_BATCH_SIZE = 32
SMOKE_SAMPLES = 8                     # spread across the train set for the estimate
MAX_ESTIMATED_HOURS = 10.0            # fail before wasting a Kaggle session
MAX_CELLS_PER_PATCH = 16              # T4-safe; keep identical across every variant
OVERWRITE = False

# Also store CellViT's raw per-pixel instance map for every patch, for
# visualising the segmentation later. No sampler reads it -- it is a sidecar,
# not part of the feature contract. Each map is zlib-compressed on its own:
# measured 227-298x on capped patches, so ~0.15 GiB per 100k patches against a
# 37 GiB raw size. Turning it off later requires re-extraction, so leaving it on
# is usually the cheaper choice.
SAVE_INSTANCE_MAPS = True

# Store ONLY CellViT's own per-cell embeddings, not the masked-crop DINOv2 ones.
#
# The cache then holds one cellvit_embeddings row per nucleus plus offsets,
# confidence, bboxes and the instance maps -- everything the sampler's default
# cell_source="cellvit_embedding" reads. Dropping the crop features removes
# about two thirds of the cache (768-d fp16 per cell against CellViT's 384-d)
# and roughly halves extraction time, since the crop encoder is a second forward
# pass over every nucleus.
#
# The cost: a run with cell_source="crop_dino" would have nothing to read and
# would need re-extraction. Set this False if that ablation is still on the table.
SKIP_CROP_DINO = True

# Use both T4s. False forces one GPU, which is the reference path.
PARALLEL = True

# Where a .npz dataset is re-exported as memory-mappable .npy files. Required
# for a 2-GPU run on pathmnist: an eager .npz read costs ~15 GiB per process and
# two workers exceed the ~30 GiB a Kaggle session has, so the second is
# OOM-killed inside np.load. Needs about as much scratch disk as the .npz.
# Unused for ImageFolder datasets (histoset, skintissue), which read per file.
MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

assert Path(DATA_PATH).exists(), DATA_PATH
assert Path(CHECKPOINT_PATH).is_file(), CHECKPOINT_PATH
assert MAGNIFICATION in (20, 40)
assert not str(CACHE_DIR).startswith("/kaggle/input"), "CACHE_DIR must be writable"

In [ ]:
from huggingface_hub import snapshot_download

# Pull the DINO weights once in the parent: two shard workers racing to populate
# the same Hugging Face cache is avoidable work and an avoidable failure mode.
if "/" in DINO_MODEL and not Path(DINO_MODEL).exists():
    print(f"Downloading {DINO_MODEL} ...")
    snapshot_download(repo_id=DINO_MODEL)

In [ ]:
# Exact one-batch integration test. Do not continue if this cell fails.
#
# It benchmarks ONE GPU, because that is what a single-batch pilot can measure.
# --shards tells it how many cards the real run will use, so the safety gate
# compares wall clock against the limit rather than GPU-hours: without it a
# 14.5 GPU-hour job that finishes in ~7 h on two T4s is rejected against a 10 h
# limit. Both numbers are printed.
#
# The pilot also runs cold-ish and measures a single batch, so it has come out
# 1.3-1.5x pessimistic against real runs (pathmnist 4.5 -> 3.0 h, histoset
# 5.2 -> 4.0 h). Treat the printed figure as an upper bound.
from utils.parallel import visible_gpu_count as _visible_gpu_count

PREFLIGHT_SHARDS = max(1, _visible_gpu_count() if PARALLEL else 1)
preflight = [
    sys.executable, "scripts/preflight_cellvit.py",
    "--dataset", DATASET, "--data_path", DATA_PATH,
    "--checkpoint", CHECKPOINT_PATH, "--cache_dir", CACHE_DIR,
    "--input_mpp", str(INPUT_MPP), "--model_mpp", str(MODEL_MPP),
    "--magnification", str(MAGNIFICATION),
    "--vit_name", DINO_MODEL, "--seed", str(SEED),
    "--smoke_samples", str(SMOKE_SAMPLES),
    "--cellvit_batch_size", str(BATCH_SIZE),
    "--dino_crop_batch_size", str(DINO_CROP_BATCH_SIZE),
    "--max_estimated_hours", str(MAX_ESTIMATED_HOURS),
    "--shards", str(PREFLIGHT_SHARDS),
]
if MAX_CELLS_PER_PATCH is not None:
    preflight += ["--max_cells_per_patch", str(MAX_CELLS_PER_PATCH)]
if SKIP_CROP_DINO:
    # So the size and runtime estimates describe the run that will happen.
    preflight.append("--skip_crop_dino")
if SAVE_INSTANCE_MAPS:
    # So the disk estimate covers the sidecar too, measured on this dataset's own
    # nucleus density rather than a guessed compression ratio.
    preflight.append("--save_instance_maps")
subprocess.check_call(preflight)

In [ ]:
import shutil

from data.npz_mmap import export_npz_to_npy
from scripts.cellvit_shard_worker import build_shard_jobs, run_cellvit_shard
from utils import nucleus_archive_stem
from utils.parallel import run_variants_parallel, visible_gpu_count

# Every flag both the shard workers and the assembly pass need. Kept in one dict
# so a shard and its assembly cannot disagree about scale or checkpoint — a
# disagreement the .build state check would reject only after the GPU time was
# already spent.
OPTIONS = {
    "dataset": DATASET,
    "data_path": DATA_PATH,
    "checkpoint": CHECKPOINT_PATH,
    "cache_dir": CACHE_DIR,
    "input_mpp": INPUT_MPP,
    "model_mpp": MODEL_MPP,
    "magnification": MAGNIFICATION,
    "vit_name": DINO_MODEL,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "dino_crop_batch_size": DINO_CROP_BATCH_SIZE,
}
if MAX_CELLS_PER_PATCH is not None:
    OPTIONS["max_cells_per_patch"] = MAX_CELLS_PER_PATCH

# Export the .npz to memory-mappable .npy ONCE, here in the parent.
#
# numpy silently ignores mmap_mode for a .npz -- NpzFile reads each member
# through zipfile -- so only standalone .npy files can actually be mapped. This
# must run in the parent: two shard workers exporting the same files would race.
# Once exported, every process maps the same pages and the OS page cache serves
# them all from one copy.
if DATA_PATH.endswith(".npz"):
    export_npz_to_npy(DATA_PATH, MMAP_CACHE_DIR)
    OPTIONS["mmap_cache_dir"] = MMAP_CACHE_DIR
    free = shutil.disk_usage(MMAP_CACHE_DIR).free
    print(f"mmap export ready | free disk {free / 2**30:.1f} GiB")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

SHARDS = max(1, visible_gpu_count() if PARALLEL else 1)
print(f"GPUs visible: {visible_gpu_count()} | shards: {SHARDS}")
print("cache:", Path(CACHE_DIR) / f"{DATASET}_seed{SEED}")

In [ ]:
import time


def free_gib(path="/kaggle/working"):
    return shutil.disk_usage(path).free / 2**30


def drop_mmap_export():
    """Delete the .npy export as soon as no process still needs pixels.

    It is ~15 GiB for PathMNIST-224 and is pure scratch: only the CellViT
    forward pass reads it. Assembly reads the .build shards instead, so holding
    the export through assembly is 15 GiB of a ~20 GB quota spent on nothing --
    which is exactly how a five-hour extraction died with
    `OSError: [Errno 28] No space left on device` at the merge step.
    """
    path = OPTIONS.get("mmap_cache_dir")
    if path and Path(path).is_dir():
        shutil.rmtree(path, ignore_errors=True)
        OPTIONS.pop("mmap_cache_dir", None)
        print(f"removed mmap export | free disk {free_gib():.1f} GiB")


manifest = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
started = time.time()
print(f"free disk before extraction: {free_gib():.1f} GiB")

if manifest.is_file() and not OVERWRITE:
    print("completed cache already exists; skipping extraction:", manifest)
    drop_mmap_export()
elif SHARDS == 1:
    run_cellvit_shard(OPTIONS, overwrite=OVERWRITE,
                      save_instance_maps=SAVE_INSTANCE_MAPS,
                      skip_crop_dino=SKIP_CROP_DINO)
else:
    # Phase 1: every GPU extracts its own block of batches into .build/.
    jobs = build_shard_jobs(OPTIONS, SHARDS, overwrite=OVERWRITE,
                            save_instance_maps=SAVE_INSTANCE_MAPS,
                            skip_crop_dino=SKIP_CROP_DINO)
    results = run_variants_parallel(jobs, run_cellvit_shard, num_workers=SHARDS)
    failed = [r["label"] for r in results if not r["ok"]]
    assert not failed, f"shards failed: {failed}"

    # Reclaim the export BEFORE assembly, not after. Assembly is the disk peak:
    # it writes the merged memmaps and then the final cache, so both exist at
    # once. Nothing past this point reads raw pixels.
    drop_mmap_export()

    # Phase 2: one process merges the shards. This rebuilds the ragged `offsets`
    # array AND concatenates the instance-map blobs, both by reading every batch
    # file in patch order -- so the result does not depend on which worker wrote
    # which file, and a missing batch is a hard error rather than a silently
    # short cache. No GPU, no models loaded.
    print(f"free disk before assembly: {free_gib():.1f} GiB")
    run_cellvit_shard(OPTIONS, assemble_only=True,
                      save_instance_maps=SAVE_INSTANCE_MAPS,
                      skip_crop_dino=SKIP_CROP_DINO)

drop_mmap_export()   # single-shard path, and a no-op when already gone
print(f"extraction total {time.time() - started:.0f}s | free disk {free_gib():.1f} GiB")


In [ ]:
# Verify the cache the sampler will actually read: load it through the same
# loader `main.py` uses, and check its sample order against a freshly built
# split. A cache whose rows are misaligned with the split loads fine and
# silently corrupts every experiment downstream, so this check is the point.
import numpy as np

from data.identity import sample_order_fingerprint
from data.loaders import get_data_loaders, get_sample_ids
from features.cellvit.cache import load_cellvit_cache
from utils import set_seed

set_seed(SEED)
# num_workers=0 here: this only needs sample IDs, not decoded pixels, so a
# worker pool would be pure startup cost.
train_loader, _, _ = get_data_loaders(
    DATA_PATH, SEED, verbose=True,
    mmap_cache_dir=OPTIONS.get("mmap_cache_dir"), num_workers=0,
)
expected_ids = get_sample_ids(train_loader.dataset)

cache = load_cellvit_cache(
    str(Path(CACHE_DIR) / f"{DATASET}_seed{SEED}"),
    expected_sample_ids=expected_ids,   # raises if the order does not match
)
assert cache.num_patches == len(expected_ids), (cache.num_patches, len(expected_ids))
assert cache.manifest["sample_fingerprint"] == sample_order_fingerprint(expected_ids)
assert cache.manifest["dataset"] == DATASET and cache.manifest["seed"] == SEED
assert cache.offsets[0] == 0 and cache.offsets[-1] == cache.num_cells

empty = int(np.sum(np.diff(cache.offsets) == 0))
print(f"OK patches={cache.num_patches} cells={cache.num_cells} "
      f"mean={cache.num_cells / max(1, cache.num_patches):.1f}/patch")
print(f"    cellvit={cache.features('cellvit_embedding').shape}")
if cache.manifest.get("has_cell_dino_features"):
    print(f"    crop_dino={cache.features('crop_dino').shape}")
else:
    print("    crop_dino: not stored (SKIP_CROP_DINO) -> "
          "the sampler must keep cell_source='cellvit_embedding'")
# The manifest must agree with what this run asked for, or the cache is not the
# one the notebook thinks it built.
assert bool(cache.manifest.get("has_cell_dino_features")) is not SKIP_CROP_DINO
print(f"    patches with no nucleus: {empty} "
      f"({100 * empty / max(1, cache.num_patches):.1f}%) -> scalpel missing_impute")

# The instance-map sidecar: read one back and check it against the cell count
# the feature arrays claim for the same patch. A blob assembled in the wrong
# order still decompresses into plausible-looking masks, so agreement between
# the two independent records is what actually catches a misalignment.
if SAVE_INSTANCE_MAPS:
    from features.cellvit.segmaps import has_instance_maps, read_instance_map

    cache_path = Path(CACHE_DIR) / f"{DATASET}_seed{SEED}"
    assert has_instance_maps(str(cache_path)), "instance maps requested but absent"
    assert cache.manifest.get("has_instance_maps") is True

    probe = next(
        (i for i in range(cache.num_patches)
         if cache.offsets[i + 1] > cache.offsets[i]),
        0,
    )
    instance_map = read_instance_map(str(cache_path), probe)
    labels = int(len(np.unique(instance_map)) - (1 if (instance_map == 0).any() else 0))
    expected = int(cache.offsets[probe + 1] - cache.offsets[probe])
    sidecar_mb = (cache_path / "instance_maps.bin").stat().st_size / 1e6
    print(f"    instance maps: {sidecar_mb:.1f} MB total "
          f"({sidecar_mb * 1000 / max(1, cache.num_patches):.1f} KB/patch)")
    print(f"    patch {probe}: map {instance_map.shape} holds {labels} instance(s), "
          f"features hold {expected} cell(s)")
    # The map keeps every nucleus CellViT found; the feature rows are capped by
    # MAX_CELLS_PER_PATCH and filtered by min_cell_area, so the map is a superset.
    assert labels >= expected, (labels, expected)
assert not (Path(CACHE_DIR) / f"{DATASET}_seed{SEED}" / ".build").exists(), (
    "the .build directory survives only when assembly did not finish"
)

# The .npy export was scratch for the extraction, not an output. Removing it
# frees the disk before the archive is written; it is rebuilt on demand.
if OPTIONS.get("mmap_cache_dir") and Path(OPTIONS["mmap_cache_dir"]).is_dir():
    freed = sum(
        f.stat().st_size
        for f in Path(OPTIONS["mmap_cache_dir"]).rglob("*") if f.is_file()
    )
    shutil.rmtree(OPTIONS["mmap_cache_dir"], ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")


In [ ]:
# Package the cache as ONE zip at the top of /kaggle/working, then delete the
# loose files.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and that is the only way to get a file out of a "Save & Run All" session
# -- there is no terminal and no kaggle CLI. So the zip goes to the top level
# where it is easy to find, and the loose cache is removed once it exists:
# keeping both doubles the download, and a session over the ~20 GB Output quota
# shows NOTHING at all. A nucleus cache is the large one, so this matters here.
#
# The name carries dataset + seed + checkpoint + crop encoder + cell cap, which
# are exactly the axes that make two nucleus caches non-interchangeable.
import json
import shutil

SOURCE = Path(CACHE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "CACHE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = nucleus_archive_stem(
    DATASET, SEED, Path(CHECKPOINT_PATH).stem, DINO_MODEL, MAX_CELLS_PER_PATCH
)
ARCHIVE = WORKING / STEM

# A successful assembly removes .build itself, so it is normally absent. It
# survives only when assembly did NOT finish -- and then it holds a full second
# copy of every per-batch shard, which would double the archive. Refuse rather
# than quietly ship a cache whose assembly never completed.
leftover_build = [p for p in SOURCE.rglob(".build") if p.is_dir()]
assert not leftover_build, (
    f"assembly did not finish: {leftover_build}. Re-run the extraction cell "
    "before archiving."
)
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

# The zip root holds `<dataset>_seed<seed>/`, the directory name the run
# notebook probes for.
print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file() and ".build" not in path.parts:
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.1f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)
if OPTIONS.get("mmap_cache_dir") and Path(OPTIONS["mmap_cache_dir"]).is_dir():
    shutil.rmtree(OPTIONS["mmap_cache_dir"], ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In run_al_sampler.ipynb: Add Data -> your new dataset. It probes for
     '{DATASET}_seed{SEED}/manifest.json' a few levels down, so there is no
     path to edit.""")
